In [1]:
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")          # reads ANTHROPIC_API_KEY from .env into the environment
client = anthropic.Anthropic()  # finds the key automatically; it never appears in your code

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(response.content[0].text)

Hello there, friend. How are you?


In [ ]:
SYSTEM = """You answer questions about board and card game rules, using ONLY the rulebook excerpts provided.

Rules:
1. Use only the excerpts. Do not use outside knowledge, even if you know the game well.
2. Each excerpt is labeled with its game. Only use excerpts from the game the question asks about.
   If the question names no game, answer separately for each game the excerpts cover, labeled by game.
   If the question uses a term that belongs to only one game (e.g. "hole cards" for Texas Hold'em), treat it as about that game.
3. After each claim, cite the excerpt id in square brackets, e.g. [catan-p9-c0].
4. If the excerpts do not contain the answer, reply exactly: "The provided rules don't cover this."
   You may then mention the closest related rule, clearly labeled as related.
5. Be short and direct."""

In [3]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":   # only move up if we're still in notebooks/
    os.chdir("..")
sys.path.append("src")
print("Working in:", os.getcwd())    # should end in /Game
from rulebook_rag.retrieve import Retriever

retriever = Retriever()

def format_context(chunks):
    parts = []
    for c in chunks:
        parts.append(f"<excerpt id=\"{c['id']}\" game=\"{c['game']}\" page=\"{c['page']}\">\n{c['text']}\n</excerpt>")
    return "\n\n".join(parts)

def answer(question, k=3):
    chunks = retriever.search(question, k=k)
    user_message = f"Excerpts:\n{format_context(chunks)}\n\nQuestion: {question}"
    response = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    text = "".join(b.text for b in response.content if b.type == "text")
    return text, chunks

Working in: /Users/yasasvikotagiri/Desktop/Game


/Users/yasasvikotagiri/Desktop/Game/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1026 17:30:23.953000 47500 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9205.92it/s]


In [4]:
for q in [
    "How many private hole cards is each player dealt?",           # direct lookup
    "Do you collect money for landing on Free Parking?",           # myth-buster
    "What penalty does the CATAN rulebook impose for flipping the board?",  # unanswerable
]:
    text, chunks = answer(q)
    print("Q:", q)
    print("A:", text)
    print("Chunks:", [c["id"] for c in chunks])
    print("=" * 60)

Q: How many private hole cards is each player dealt?
A: **Texas Hold'em:** Each player is dealt two private hole cards [texas_holdem-p2-c1]. The deal rule states that every player receives two cards for their eyes only [texas_holdem-p2-c0].

**Rummy:** The provided rules don't cover this. Related: in rummy, the number of cards dealt depends on the number of players, for example 7 cards with 3 or 4 players [rummy-p4-c1]. Rummy has no private hole cards in these excerpts.
Chunks: ['texas_holdem-p2-c1', 'rummy-p4-c1', 'texas_holdem-p2-c0']
Q: Do you collect money for landing on Free Parking?
A: No. Landing on Free Parking gives you no bonus: "you don't get a bonus for landing on Free Parking!" [monopoly-p1-c0]. The space itself says "Relax! Nothing happens." [monopoly-p2-c5]
Chunks: ['monopoly-p1-c0', 'monopoly-p2-c5', 'monopoly-p2-c3']
Q: What penalty does the CATAN rulebook impose for flipping the board?
A: The provided rules don't cover this.

Related (setup only): The variable setup d